# Giai đoạn 3: Xây dựng và Đánh giá Mô hình Hồi quy
Notebook này thực hiện việc chia tách dữ liệu, chuẩn hóa đặc trưng, huấn luyện các thuật toán hồi quy chính (Linear Regression, Polynomial Regression, Random Forest, Gradient Boosting) và thực hiện phân tích chuyên sâu lỗi dự báo (Residual Analysis).

In [1]:
# ==========================================
# IMPORT THƯ VIỆN VÀ CHUẨN BỊ DỮ LIỆU
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# 1. Đọc dữ liệu sạch
try:
    df = pd.read_csv('../data/processed/cleaned_crime_data_183.csv')
except FileNotFoundError:
    df = pd.read_csv('cleaned_crime_data_183.csv')

TARGET = 'ViolentCrimesPerPop'

# 2. Tách đặc trưng và biến mục tiêu
X = df.drop(columns=[TARGET])
y = df[TARGET]
X = X.select_dtypes(include=[np.number])

# 3. Phân chia tập huấn luyện và tập kiểm thử (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Chuẩn hóa đặc trưng (Feature Scaling)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Số lượng mẫu huấn luyện (Train): {X_train.shape[0]}")
print(f"Số lượng mẫu kiểm thử (Test): {X_test.shape[0]}")

Số lượng mẫu huấn luyện (Train): 1595
Số lượng mẫu kiểm thử (Test): 399


## 1. Huấn luyện các mô hình và Đánh giá Hiệu năng

In [2]:
# Khởi tạo các mô hình hồi quy
models = {
    "Linear Regression": LinearRegression(),
    "Polynomial Regression (Bậc 2)": make_pipeline(PolynomialFeatures(degree=2), Ridge(alpha=10.0)),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting Regressor": GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = []
predictions = {}

for name, model in models.items():
    # Huấn luyện mô hình
    model.fit(X_train_scaled, y_train)
    
    # Dự đoán
    y_pred = model.predict(X_test_scaled)
    
    # Ràng buộc giá trị dự đoán không âm
    y_pred = np.clip(y_pred, 0, None)
    predictions[name] = y_pred
    
    # Tính chỉ số đánh giá
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    results.append({
        "Mô hình": name,
        "R2 Score": r2,
        "RMSE": rmse,
        "MAE": mae
    })

# Hiển thị bảng kết quả so sánh
results_df = pd.DataFrame(results).sort_values(by="R2 Score", ascending=False)
print("--- BẢNG SO SÁNH HIỆU NĂNG CÁC MÔ HÌNH ---")
display(results_df)

--- BẢNG SO SÁNH HIỆU NĂNG CÁC MÔ HÌNH ---


,Mô hình,R2 Score,RMSE,MAE
0,Linear Regression,0.639985,0.131313,0.093527
2,Random Forest Regressor,0.618040,0.135256,0.092320
3,Gradient Boosting Regressor,0.599244,0.138544,0.091079
1,Polynomial Regression (Bậc 2),0.091975,0.208544,0.144153


## 2. Phân tích mức độ quan trọng của đặc trưng (Feature Importance)

In [ ]:
# Xác định mô hình tốt nhất dựa trên chỉ số R2 Score
best_model_name = results_df.iloc[0]['Mô hình']
best_model = models[best_model_name]

plt.figure(figsize=(12, 6))

if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=X.columns)
    top_10 = importances.sort_values(ascending=False).head(10)
    sns.barplot(x=top_10.values, y=top_10.index, palette='viridis')
    plt.title(f'Top 10 Đặc trưng thúc đẩy hiệu năng mô hình tốt nhất ({best_model_name})', fontweight='bold', fontsize=13)
elif hasattr(best_model, 'coef_'):
    coefs = pd.Series(best_model.coef_, index=X.columns)
    top_10 = coefs.abs().sort_values(ascending=False).head(10)
    sns.barplot(x=top_10.values, y=top_10.index, palette='mako')
    plt.title(f'Top 10 Hệ số có trọng số lớn nhất ({best_model_name})', fontweight='bold', fontsize=13)
else:
    print(f"Mô hình {best_model_name} (Pipeline) cần trích xuất đặc trưng phức tạp hơn, hãy tham khảo các chỉ số R2/RMSE ở trên.")

plt.xlabel('Trọng số đóng góp')
plt.ylabel('Tên đặc trưng')
plt.tight_layout()
plt.show()

## 3. Phân tích phần dư (Residuals Analysis) nhằm kiểm tra giả định mô hình

In [ ]:
best_pred = predictions[best_model_name]
residuals = y_test - best_pred

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Đồ thị 1: Thực tế vs Dự đoán
sns.scatterplot(x=y_test, y=best_pred, alpha=0.6, ax=axes[0], color='indigo')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_title(f'Giá trị Thực tế vs Dự đoán ({best_model_name})', fontweight='bold')
axes[0].set_xlabel('Giá trị Thực tế (ViolentCrimesPerPop)')
axes[0].set_ylabel('Giá trị Dự đoán (ViolentCrimesPerPop)')

# Đồ thị 2: Residual Plot
sns.scatterplot(x=best_pred, y=residuals, alpha=0.6, ax=axes[1], color='darkorange')
axes[1].axhline(y=0, color='red', linestyle='--', lw=2)
axes[1].set_title('Biểu đồ phân tán Phần dư (Residual Plot)', fontweight='bold')
axes[1].set_xlabel('Giá trị dự đoán')
axes[1].set_ylabel('Phần dư (Sai số)')

plt.tight_layout()
plt.show()